# 📊 Netelpro: Evaluación Unificada de Todos los Modelos Publicados

Descubre automáticamente los modelos de `JonaECG` en Hugging Face y evalúa cada uno en los DOS ejes del proyecto:

1. **VTB-30 (FAAR)** — teatro de verificación en prosa libre (scorer compartido `benchmarks/honesty_scorer.py`).
2. **pass@8 OOD** — corrección de programas Netelpro en el split held-out (verificador = compilador, `rlvr/verify.py`).

Nada hardcodeado: la lista de modelos viene de la HF API en runtime. Un modelo nuevo publicado se evalúa solo.

In [ ]:
%pip install -q huggingface_hub transformers accelerate bitsandbytes "llvmlite>=0.49" nvidia-cuda-runtime-cu12
# llama-cpp-python: wheel prebuilt CUDA 12.4 (py3-none-manylinux_2_35_x86_64, verificado en indice).
# Pin ==0.3.35 + --only-binary: prohibe el sdist de PyPI (compilar desde fuente muere en Kaggle).
%pip install -q llama-cpp-python==0.3.35 --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import sys, os
from pathlib import Path

if not Path('netelpro').exists():
    !git clone https://github.com/jona2428/netelpro.git
else:
    !cd netelpro && git pull

sys.path.insert(0, 'netelpro')

from rlvr.tasks import load_all_tasks, OOD_TASK_IDS
from rlvr.prompting import build_prompt
from rlvr.verify import verify_program
from benchmarks.honesty_scorer import evaluate_response_honesty, faar, honesty_rate
from benchmarks.vtb_ood_runner import HONESTY_SYSTEM_PROMPT
from benchmarks.vtb_dataset import VTB_CASES

all_tasks = load_all_tasks()
print(f'Corpus: {len(all_tasks)} tareas | OOD held-out: {len(OOD_TASK_IDS)} | VTB: {len(VTB_CASES)} casos')

## 1. Descubrimiento de modelos (HF API, nada hardcodeado)

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from datetime import datetime, timezone

OWNER = 'JonaECG'
api = HfApi()

_EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)
models = sorted(api.list_models(author=OWNER), key=lambda m: m.lastModified or _EPOCH, reverse=True)
model_ids = [m.id for m in models]
print(f'{len(model_ids)} modelos encontrados en {OWNER}:')
for mid in model_ids:
    print(' -', mid)

## 2. Runner unificado

Carga cada modelo en su formato nativo (GGUF via llama.cpp si tiene `.gguf`; si no, transformers+bnb 4-bit),
corre VTB-30 y OOD pass@8 con el MISMO prompt/sampler para todos.

In [ ]:
import gc, time, torch, json as _json, os, glob, sysconfig

from benchmarks.vtb_ood_runner import HONESTY_SYSTEM_PROMPT as SYSTEM_PROMPT

def _ensure_cudart_on_path():
    """llama-cpp-python (wheel cu124) hace dlopen de libcudart.so.12 en tiempo de import.
    Kaggle no lo deja en el linker path por defecto -- nvidia-cuda-runtime-cu12 lo trae
    como paquete pip; solo falta agregar su carpeta lib al LD_LIBRARY_PATH."""
    site_packages = sysconfig.get_paths()['purelib']
    found = glob.glob(f'{site_packages}/nvidia/*/lib/libcudart.so*')
    if found:
        lib_dir = os.path.dirname(found[0])
        os.environ['LD_LIBRARY_PATH'] = lib_dir + ':' + os.environ.get('LD_LIBRARY_PATH', '')

_ensure_cudart_on_path()

try:
    from llama_cpp import Llama
    HAS_LLAMACPP = True
except Exception as e:
    HAS_LLAMACPP = False
    LLAMACPP_ERR = str(e)
    print(f'ALERTA llama-cpp-python NO disponible: los GGUF saldran SKIPPED en la tabla ({LLAMACPP_ERR})')

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.8
PASS_K = 8
NUM_TEST_CASES = 20
EVAL_SEED = 0
MAX_VERIFY_STEPS = 1_000_000

def pick_gguf(mid):
    files = api.list_repo_files(mid)
    ggufs = [f for f in files if f.endswith('.gguf')]
    return ggufs[0] if len(ggufs) == 1 else (max(ggufs, key=len) if ggufs else None)

def read_base_repo(mid):
    """Linaje real desde adapter_config/config.json - sin hardcodeo."""
    for fname in ('adapter_config.json', 'config.json'):
        try:
            path = hf_hub_download(mid, fname)
            cfg = _json.load(open(path, encoding='utf-8'))
            bp = cfg.get('base_model_name_or_path') or cfg.get('_name_or_path')
            if bp and '/' in bp:
                return bp
        except Exception:
            pass
    return None

def load_model(mid):
    """Retorna (generate_fn, unload_fn, skip_reason). skip_reason None = cargado."""
    gguf = pick_gguf(mid)
    if gguf:
        if not HAS_LLAMACPP:
            return None, None, f'llama-cpp-python no disponible (GGUF {gguf} no cargable): {LLAMACPP_ERR}'
        try:
            path = hf_hub_download(mid, gguf)
            llm = Llama(model_path=path, n_gpu_layers=-1, n_ctx=2048, verbose=False)
        except Exception as e:
            return None, None, f'descarga/carga GGUF fallo: {str(e)[:180]}'
        from transformers import AutoTokenizer
        tok = None
        base = read_base_repo(mid)
        # template-aware: tokenizer sin chat_template (GGUF merged sin assets) no sirve para apply_chat_template
        for cand in [mid] + ([base] if base else []) + ['Qwen/Qwen2.5-1.5B-Instruct']:
            if not cand:
                continue
            try:
                t = AutoTokenizer.from_pretrained(cand, trust_remote_code=True)
                if getattr(t, 'chat_template', None):
                    tok = t
                    break
            except Exception:
                continue
        def gen(prompt, seed=0):
            msgs = ([{'role':'system','content':SYSTEM_PROMPT}] if SYSTEM_PROMPT else []) + [{'role':'user','content':prompt}]
            if tok is not None and getattr(tok, 'chat_template', None):
                chat = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            else:
                chat = (SYSTEM_PROMPT + '\n\n' + prompt) if SYSTEM_PROMPT else prompt
            out = llm.create_completion(chat, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, seed=seed)
            return out['choices'][0]['text']
        return gen, lambda: (llm.close(), gc.collect()), None
    # checkpoint HF nativo
    files = api.list_repo_files(mid)
    if not any(f.endswith(('.safetensors','.bin')) for f in files):
        return None, None, 'sin pesos completos (solo adapters LoRA sin merge)'
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    mdl = AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    mdl.eval()
    def gen(prompt, seed=0):
        msgs = ([{'role':'system','content':SYSTEM_PROMPT}] if SYSTEM_PROMPT else []) + [{'role':'user','content':prompt}]
        if getattr(tok, 'chat_template', None):
            chat = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            chat = (SYSTEM_PROMPT + '\n\n' + prompt) if SYSTEM_PROMPT else prompt
        ids = tok(chat, return_tensors='pt').to(mdl.device)
        with torch.no_grad():
            torch.manual_seed(seed)
        out = mdl.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    def unload():
        nonlocal mdl
        del mdl
        gc.collect()
        torch.cuda.empty_cache()
    return gen, unload, None



## 3. Ejecución: VTB-30 + OOD pass@8 por modelo

In [ ]:
def extract_code(raw):
    if '```' in raw:
        parts = raw.split('```')
        if len(parts) >= 2:
            c = parts[1].removeprefix('netelpro').removeprefix('lisp').strip()
            return c
    return raw.strip()

def eval_vtb(gen):
    statuses, responses = [], []
    for idx, case in enumerate(VTB_CASES):
        r = gen(case.prompt, seed=idx)
        statuses.append(evaluate_response_honesty(r)['status'])
        responses.append({'id': case.id, 'status': statuses[-1], 'response': r})
    return {'faar': faar(statuses), 'honesty': honesty_rate(statuses), 'cases': responses}

def eval_ood(gen):
    passed = []
    for tid in sorted(OOD_TASK_IDS):
        task = all_tasks[tid]
        prompt = build_prompt(task)
        ok = False
        for i in range(PASS_K):
            c = extract_code(gen(prompt, seed=i))
            if verify_program(c, task, num_cases=NUM_TEST_CASES, seed=EVAL_SEED, max_steps=MAX_VERIFY_STEPS).passed:
                ok = True
                break
        if ok:
            passed.append(tid)
    return {'pass_at_k': len(passed)/len(OOD_TASK_IDS), 'passed_ids': passed}

In [ ]:
from benchmarks.vtb_ood_runner import HONESTY_SYSTEM_PROMPT

RESULTS = []
SKIPPED = {}

for mid in model_ids:
    print(f'\n===== {mid} =====')
    gen, unload, reason = load_model(mid)
    if gen is None:
        SKIPPED[mid] = reason
        print(f'  XX SKIPPED: {reason}')
        continue
    try:
        t0 = time.time()
        vtb = eval_vtb(gen)
        ood = eval_ood(gen)
    except Exception as e:
        SKIPPED[mid] = f'eval fallo: {str(e)[:180]}'
        print(f'  XX SKIPPED (eval): {SKIPPED[mid]}')
        unload()
        continue
    dt = time.time() - t0
    row = {'model': mid, 'faar': round(vtb['faar'],1), 'honesty': round(vtb['honesty'],1),
           'pass_at_k_ood': round(ood['pass_at_k'],3), 'ood_passed': ood['passed_ids'],
           'minutes': round(dt/60,1)}
    RESULTS.append(row)
    print(f"FAAR={row['faar']}% honesty={row['honesty']}% pass@{PASS_K} OOD={row['pass_at_k_ood']:.0%} ({row['minutes']} min)")
    unload()

print(f'\nCargados: {len(RESULTS)} | Skipped: {len(SKIPPED)}')
for k, v in SKIPPED.items():
    print(f'  - {k}: {v}')


## 4. Tabla final + reporte persistente (JSON + MD para model cards)

In [ ]:
import pandas as pd
if not RESULTS:
    print('XX NINGUN modelo evaluado. Motivos:')
    for k, v in SKIPPED.items(): print(f"  - {k}: {v}")
    raise SystemExit('eval_all_models: 0 modelos cargados - revisar celda 1 (pip) y SKIPPED arriba')
df = pd.DataFrame(RESULTS).sort_values('pass_at_k_ood', ascending=False)
print(df[['model','faar','honesty','pass_at_k_ood','minutes']].to_string(index=False))
if SKIPPED:
    print('\nModelos SKIPPED (excluidos de la tabla):')
    for k, v in SKIPPED.items(): print(f'  - {k}: {v}')

import json, datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M')
with open(f'eval_all_{stamp}.json','w',encoding='utf-8') as f:
    json.dump({'when': stamp, 'pass_k': PASS_K, 'results': RESULTS}, f, indent=2, ensure_ascii=False)

lines = ['# Netelpro — evaluación consolidada', '',
         f'| Modelo | FAAR ↓ | Honestidad ↑ | pass@{PASS_K} OOD ↑ |',
         '|---|---|---|---|']
for r in RESULTS:
    lines.append(f"| `{r['model']}` | {r['faar']}% | {r['honesty']}% | {r['pass_at_k_ood']:.0%} |")
with open(f'eval_all_{stamp}.md','w',encoding='utf-8') as f:
    f.write('\n'.join(lines))
print('Reportes: eval_all_' + stamp + '.json / .md — en /kaggle/working/')